# 面试问题：梯度累积、混合精度 Loss Scaling 与梯度裁剪应该怎样正确组合？

**一句话回答**：先明确目标是“大 batch 的样本均值梯度”。每个 micro-batch 按样本数累加 loss，窗口结束后再归一化；AMP 中先缩放 loss 做 backward，再检查溢出、反缩放梯度，随后裁剪并原子地 `step/zero_grad`。任一 rank 溢出时所有 rank 都应跳过同一次更新。

本 Notebook 用 PyTorch 基础张量和手写状态机验证等价性，不调用 `GradScaler` 或现成梯度裁剪函数。

In [ ]:
import copy, hashlib, json, math  # 导入本单元所需的依赖。
import numpy as np  # 导入本单元所需的依赖。
import torch  # 导入本单元所需的依赖。

SEED98=9801; torch.manual_seed(SEED98); rng98=np.random.default_rng(SEED98)  # 计算并保存当前步骤的中间状态。
x98=torch.randn(11,3,dtype=torch.float64); y98=x98@torch.tensor([1.2,-.7,2.],dtype=torch.float64)+.3  # 计算并保存当前步骤的中间状态。
assert x98.shape==(11,3) and y98.shape==(11,)  # 用受控断言验证关键不变量。
assert torch.isfinite(x98).all()  # 用受控断言验证关键不变量。
assert SEED98==9801  # 用受控断言验证关键不变量。

## 1. 大 batch 等价性的数学合同

若目标 loss 是 `N` 个样本的均值，micro-batch 大小为 `nᵢ`，正确做法是累计每批 **loss 之和**，最后除以 `N`。简单把每个 micro-batch 的 mean 再除以批次数，只在各批等大时成立，尾批较小时会被过度加权。

In [ ]:
w_full98=torch.tensor([.2,-.1,.4],dtype=torch.float64,requires_grad=True); b_full98=torch.tensor(0.,dtype=torch.float64,requires_grad=True)  # 计算并保存当前步骤的中间状态。
full_loss98=((x98@w_full98+b_full98-y98)**2).mean(); full_loss98.backward(); ref_w98=w_full98.grad.clone(); ref_b98=b_full98.grad.clone()  # 计算并保存当前步骤的中间状态。
w_acc98=w_full98.detach().clone().requires_grad_(); b_acc98=b_full98.detach().clone().requires_grad_()  # 计算并保存当前步骤的中间状态。
for sl in (slice(0,4),slice(4,8),slice(8,11)): (((x98[sl]@w_acc98+b_acc98-y98[sl])**2).sum()/len(x98)).backward()  # 遍历输入元素以累积或检查结果。
assert torch.allclose(w_acc98.grad,ref_w98,atol=1e-12)  # 用受控断言验证关键不变量。
assert torch.allclose(b_acc98.grad,ref_b98,atol=1e-12)  # 用受控断言验证关键不变量。
assert math.isclose(float(full_loss98),float(((x98@w_acc98+b_acc98-y98)**2).mean()))  # 用受控断言验证关键不变量。

## 2. 尾批反例与 `zero_grad` 边界

把三个 micro-batch 的 mean 平均，相当于让 3 个尾样本与前面每组 4 个样本拥有同样总权重。另一个常见错误是在每个 micro-batch 前清梯度，这会只保留最后一批。清梯度应发生在累积窗口开始或成功/跳过更新之后。

In [ ]:
w_bad98=w_full98.detach().clone().requires_grad_(); b_bad98=b_full98.detach().clone().requires_grad_()  # 计算并保存当前步骤的中间状态。
for sl in (slice(0,4),slice(4,8),slice(8,11)): (((x98[sl]@w_bad98+b_bad98-y98[sl])**2).mean()/3).backward()  # 遍历输入元素以累积或检查结果。
w_last98=w_full98.detach().clone().requires_grad_()  # 计算并保存当前步骤的中间状态。
for sl in (slice(0,4),slice(4,8),slice(8,11)):  # 遍历输入元素以累积或检查结果。
    w_last98.grad=None; ((x98[sl]@w_last98-y98[sl])**2).mean().backward()  # 计算并保存当前步骤的中间状态。
assert not torch.allclose(w_bad98.grad,ref_w98,atol=1e-6)  # 用受控断言验证关键不变量。
assert torch.allclose(w_last98.grad,2*x98[8:].T@(x98[8:]@w_last98-y98[8:])/3)  # 用受控断言验证关键不变量。
assert torch.linalg.vector_norm(w_bad98.grad-ref_w98)>1e-3  # 用受控断言验证关键不变量。

## 3. 一次更新只能发生在同步边界

梯度累积改变的是 backward 次数，不应改变 optimizer step 数。下面手写一次 SGD，验证完整 batch 与三个 micro-batch 得到完全相同参数；分布式训练中前几个 backward 通常放在 `no_sync`，最后一次才 AllReduce。

In [ ]:
lr98=.05; wf98=w_full98.detach().clone(); bf98=b_full98.detach().clone(); wa98=w_acc98.detach().clone(); ba98=b_acc98.detach().clone()  # 计算并保存当前步骤的中间状态。
wf98-=lr98*ref_w98; bf98-=lr98*ref_b98; wa98-=lr98*w_acc98.grad; ba98-=lr98*b_acc98.grad  # 计算并保存当前步骤的中间状态。
assert torch.equal(wf98,wa98) and torch.equal(bf98,ba98)  # 用受控断言验证关键不变量。
assert not torch.equal(wf98,w_full98.detach())  # 用受控断言验证关键不变量。
assert torch.isfinite(wf98).all() and torch.isfinite(bf98)  # 用受控断言验证关键不变量。

## 4. 多卡时要同时核对样本权重与通信

DDP 默认对各 rank 梯度求平均。若各 rank 有效 token 数不同，直接平均 rank mean 并非全局 token mean；应累计 loss sum 与有效计数，再按全局计数缩放。下面用 NumPy 模拟两个 rank，展示按计数加权才与拼接数据一致。

In [ ]:
g1_98=np.array([2.,4.]); g2_98=np.array([-1.,5.]); n1_98,n2_98=8,2  # 计算并保存当前步骤的中间状态。
naive_rank_mean98=(g1_98+g2_98)/2; global_sample_mean98=(n1_98*g1_98+n2_98*g2_98)/(n1_98+n2_98)  # 计算并保存当前步骤的中间状态。
concat_equiv98=np.vstack([np.repeat(g1_98[None],n1_98,0),np.repeat(g2_98[None],n2_98,0)]).mean(0)  # 计算并保存当前步骤的中间状态。
assert np.allclose(global_sample_mean98,concat_equiv98)  # 用受控断言验证关键不变量。
assert not np.allclose(naive_rank_mean98,global_sample_mean98)  # 用受控断言验证关键不变量。
assert np.allclose(global_sample_mean98,[1.4,4.2])  # 用受控断言验证关键不变量。

## 5. Loss Scaling 为什么能救 FP16 下溢

FP16 的动态范围有限，小梯度可能在 backward 中舍入为 0。先乘较大 scale，使中间梯度落入可表示区间；通信和裁剪前再除回 scale。它不会改变数学梯度，只改变有限精度下的表示路径。

In [ ]:
tiny98=1e-8; direct98=np.float16(tiny98); scale98=2**15; scaled98=np.float16(tiny98*scale98); recovered98=float(scaled98)/scale98  # 计算并保存当前步骤的中间状态。
assert direct98==0  # 用受控断言验证关键不变量。
assert scaled98>0 and recovered98>0  # 用受控断言验证关键不变量。
assert abs(recovered98-tiny98)/tiny98<.01  # 用受控断言验证关键不变量。

## 6. 动态 Scaler 是“检测—提交”事务

反缩放后检查所有梯度；只要一个非有限值，就不能部分更新参数，应整体跳过并降低 scale。连续若干次有限更新后可增长 scale。分布式场景还要对 overflow flag 做全局 OR，保证各 rank step 对齐。

In [ ]:
class DynamicScaler98:  # 定义承载本节状态与行为的数据结构。
    def __init__(self,scale=128.,growth_interval=2): self.scale=float(scale); self.good=0; self.growth_interval=growth_interval  # 定义本节可复用的核心函数。
    def unscale_and_check(self,grads):  # 定义本节可复用的核心函数。
        unscaled=[g/self.scale for g in grads]; return unscaled,all(torch.isfinite(g).all().item() for g in unscaled)  # 计算并保存当前步骤的中间状态。
    def update(self,finite):  # 定义本节可复用的核心函数。
        if not finite: self.scale=max(self.scale/2,1.); self.good=0  # 按当前条件选择后续控制路径。
        else:  # 处理前置条件不成立的分支。
            self.good+=1  # 计算并保存当前步骤的中间状态。
            if self.good==self.growth_interval: self.scale*=2; self.good=0  # 按当前条件选择后续控制路径。
scaler98=DynamicScaler98(); ug98,ok98=scaler98.unscale_and_check([torch.tensor([256.])]); scaler98.update(ok98); scaler98.update(True)  # 计算并保存当前步骤的中间状态。
assert ok98 and torch.equal(ug98[0],torch.tensor([2.]))  # 用受控断言验证关键不变量。
assert scaler98.scale==256 and scaler98.good==0  # 用受控断言验证关键不变量。
_,ok_bad98=scaler98.unscale_and_check([torch.tensor([float("inf")])]); scaler98.update(ok_bad98); assert not ok_bad98 and scaler98.scale==128  # 计算并保存当前步骤的中间状态。

## 7. Global-norm clipping 必须在反缩放之后

先计算所有参数梯度拼接后的 L2 norm，系数为 `min(1, max_norm/(norm+ε))`。若对尚未反缩放的梯度裁剪，阈值会随 scale 改变；若逐张量裁剪，则方向也不同。裁剪解决瞬时爆炸，不应掩盖持续异常。

In [ ]:
def clip_global_norm98(grads,max_norm,eps=1e-12):  # 定义本节可复用的核心函数。
    total=torch.sqrt(sum((g.double()**2).sum() for g in grads)); coef=min(1.,max_norm/(float(total)+eps)); return [g*coef for g in grads],float(total),coef  # 计算并保存当前步骤的中间状态。
raw98=[torch.tensor([3.,4.]),torch.tensor([12.])]; clipped98,norm98,coef98=clip_global_norm98(raw98,6.5)  # 计算并保存当前步骤的中间状态。
new_norm98=math.sqrt(sum(float((g**2).sum()) for g in clipped98))  # 计算并保存当前步骤的中间状态。
assert math.isclose(norm98,13.) and math.isclose(coef98,.5,rel_tol=1e-12)  # 用受控断言验证关键不变量。
assert math.isclose(new_norm98,6.5,rel_tol=1e-6)  # 用受控断言验证关键不变量。
assert torch.allclose(clipped98[0],torch.tensor([1.5,2.]))  # 用受控断言验证关键不变量。

## 8. 把顺序固化成可测试的更新函数

完整顺序是：累计 scaled loss → 到达边界 → unscale → 全局 overflow 判定 → clip → step 或 skip → zero → 更新 scale。checkpoint 最好保存在更新边界；若必须保存半个窗口，还要保存当前 `.grad`、累计样本数与 micro-step。

In [ ]:
p98=torch.tensor([1.,-2.]); scaled_grads98=[torch.tensor([384.,512.])]; scaler_loop98=DynamicScaler98(scale=128.,growth_interval=100)  # 计算并保存当前步骤的中间状态。
unscaled98,finite98=scaler_loop98.unscale_and_check(scaled_grads98); clipped_loop98,pre_norm98,_=clip_global_norm98(unscaled98,4.)  # 计算并保存当前步骤的中间状态。
before_p98=p98.clone()  # 计算并保存当前步骤的中间状态。
if finite98: p98-=.1*clipped_loop98[0]  # 按当前条件选择后续控制路径。
scaler_loop98.update(finite98)  # 执行当前语句以推进本节示例。
manifest98={"micro_batches":3,"reduction":"global_sample_mean","unscale_before_clip":True,"overflow_policy":"atomic_skip"}; digest98=hashlib.sha256(json.dumps(manifest98,sort_keys=True).encode()).hexdigest()  # 计算并保存当前步骤的中间状态。
assert finite98 and math.isclose(pre_norm98,5.)  # 用受控断言验证关键不变量。
assert torch.allclose(p98,before_p98-.1*torch.tensor([2.4,3.2]))  # 用受控断言验证关键不变量。
assert len(digest98)==64 and manifest98["unscale_before_clip"]  # 用受控断言验证关键不变量。

## 面试总结

回答时先写清目标 reduction，再给顺序：**按有效样本累积 → 最后一次同步 → unscale → 全局检查 overflow → global-norm clip → 原子 step/skip → 清梯度与更新 scale**。追问 DDP 尾批、为何 loss 要除、为什么裁剪放在 unscale 后、checkpoint 能否落在半窗口，都可由同一状态机回答。

延伸阅读：[PyTorch AMP Examples](https://pytorch.org/docs/stable/notes/amp_examples.html)、[Automatic Mixed Precision](https://developer.nvidia.com/automatic-mixed-precision)、[DDP 文档](https://pytorch.org/docs/stable/generated/torch.nn.parallel.DistributedDataParallel.html)。